# MAMBA

> MAMBA (model-based MARL with communication) on MultiGrid: the paper's version, and a goal-conditioned one.

In [ ]:
#| default_exp wm.mamba.core

In [ ]:
#| hide
from nbdev.showdoc import *

MAMBA (Egorov & Shpilman, AAMAS 2022) on MultiGrid envs, in two versions:

| version | goal | reward | as in |
|---|---|---|---|
| `Mamba(goal_conditioned=False)` | not given (found by exploring) | the env's reward, plus MAMBA's Flatland shaping: `finish_value` when an agent reaches its goal and `near_coeff` per cell of progress along the shortest path | the paper (its Flatland setup) |
| `Mamba(goal_conditioned=True)` | each agent's offset to its goal cell (a compass) | `finish_value` when an agent reaches its goal; every episode is also stored again with hindsight goals (`relabel_k` copies: cells the agents reached later) | not in the paper (GC-MAMBA) |

MAMBA learns **online**: it alternates playing an episode and training on its replay buffer, its
world model (a DreamerV2 RSSM per agent, shared weights, attending over the agents) and its actor and
critic (PPO in imagination). Its transition attends over all the agents, at training and at execution:
the agents communicate their latent states. The vendored code (`stable_marl.wm.mamba.mamba_marl`, see its
README) is the paper's; this module replaces its SMAC / Flatland workers with a loop over a `World`.

Observations are vectors (MAMBA's encoders are MLPs): by default each agent's symbolic local view
(`image`, in the env's `obs_mode`) and heading (`direction`). Agents that already reached their goal
see MAMBA's absorbing (zero) observation, as in its workers.

In [ ]:
#| export
from __future__ import annotations

import itertools
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import torch

from stable_marl.policy import BasePolicy

### Observations

In [ ]:
#| export
_IMAGE_SCALE = 10.0   # symbolic cells (type, color, state) scaled to about [0, 1]


def _feature(key: str, value: np.ndarray, grid_size: int) -> np.ndarray:
    "One info value of one agent as a flat float vector."
    value = np.asarray(value)
    if key == 'direction':
        return np.eye(4, dtype=np.float32)[int(value)]
    if key == 'image':
        return value.astype(np.float32).reshape(-1) / _IMAGE_SCALE
    if key == 'pixels':
        return value.astype(np.float32).reshape(-1) / 255.0
    if key == 'position':
        return value.astype(np.float32).reshape(-1) / grid_size
    return value.astype(np.float32).reshape(-1)


def agent_features(info: dict, i: int, keys: Sequence[str], grid_size: int,
                   goal: np.ndarray | None = None) -> np.ndarray:
    """
    Observations ``(num_agents, dim)`` of env `i` in stacked infos ``(num_envs, 1, num_agents, ...)``:
    the `keys` of each agent, plus its offset to `goal` ``(num_agents, 2)`` if given.
    """
    num_agents = info[keys[0]].shape[2]
    rows = []
    for a in range(num_agents):
        parts = [_feature(k, info[k][i, 0, a], grid_size) for k in keys]
        if goal is not None:
            parts.append((np.asarray(goal[a], np.float32) - info['position'][i, 0, a]) / grid_size)
        rows.append(np.concatenate(parts))
    return np.stack(rows).astype(np.float32)

### Training

`Mamba.train` plays `episodes` episodes in env 0 of a `World` (made with `num_envs=1`), each followed by
MAMBA's learner step (`DreamerLearner.step`: a world model update and PPO in imagination, once its buffer
holds `MIN_BUFFER_SIZE` steps). `config` overrides MAMBA's settings (`DreamerLearnerConfig`, upper-case
names, e.g. ``HIDDEN=128, MIN_BUFFER_SIZE=500``).

In [ ]:
#| export
_DEFAULTS = dict(CAPACITY=100_000)   # MAMBA: 500k steps; fewer for a CPU


class Mamba:
    """
    MAMBA's learner and controller for `num_agents` agents with `num_actions` discrete actions.

    Parameters
    ----------
    goal_conditioned : bool
        GC-MAMBA (goal offset in the observations, goal-reaching reward, hindsight relabelling), or
        MAMBA with its shaped reward
    obs_keys : list of str
        Info keys of each agent's observation (``image``, ``direction``, ``position``, ``pixels``, ...)
    grid_size : int
        Size of the grid (scales positions and goal offsets)
    finish_value : float
        Reward for reaching the goal (MAMBA's Flatland ``FinishRewardConfig(finish_value=10)``)
    near_coeff : float
        Reward per cell of progress towards the goal (``NearRewardConfig(coeff=0.01)``; MAMBA only)
    relabel_k : int
        Hindsight copies of each episode (GC-MAMBA only)
    seed : int
        Seed of torch and numpy (MAMBA samples with their global generators)
    **config
        MAMBA settings (``DreamerLearnerConfig`` attributes)
    """
    def __init__(self, num_agents: int, num_actions: int, goal_conditioned: bool = False,
                 obs_keys: Sequence[str] = ('image', 'direction'), grid_size: int = 15, finish_value: float = 10.0,
                 near_coeff: float = 0.01, relabel_k: int = 4, seed: int = 0, obs_dim: int | None = None, **config):
        from stable_marl.wm.mamba.mamba_marl.configs.dreamer.DreamerControllerConfig import DreamerControllerConfig
        from stable_marl.wm.mamba.mamba_marl.configs.dreamer.DreamerLearnerConfig import DreamerLearnerConfig
        from stable_marl.wm.mamba.mamba_marl.environments import Env
        self.kwargs = dict(num_agents=num_agents, num_actions=num_actions, goal_conditioned=goal_conditioned,
                           obs_keys=list(obs_keys), grid_size=grid_size, finish_value=finish_value, near_coeff=near_coeff,
                           relabel_k=relabel_k, seed=seed, obs_dim=obs_dim, **config)
        self.num_agents, self.num_actions, self.goal_conditioned = num_agents, num_actions, goal_conditioned
        self.obs_keys, self.grid_size = list(obs_keys), grid_size
        self.finish_value, self.near_coeff, self.relabel_k = finish_value, near_coeff, relabel_k
        self.obs_dim = obs_dim
        self.config = config
        self.seed = seed
        self._configs = (DreamerLearnerConfig, DreamerControllerConfig, Env)
        self.learner = None
        if obs_dim is not None:
            self._build(obs_dim)

    def _apply(self, cfg, strict: bool = True):
        "MAMBA's config `cfg` for our env, with our settings (`strict`: each must exist; else those that do)."
        _, _, Env = self._configs
        cfg.IN_DIM, cfg.ACTION_SIZE, cfg.ENV_TYPE = self.obs_dim, self.num_actions, Env.FLATLAND   # no action masks
        for key, value in {**_DEFAULTS, **self.config}.items():
            if not hasattr(cfg, key):
                if strict:
                    raise KeyError(f"MAMBA has no setting {key!r}")
                continue
            setattr(cfg, key, value)
        if 'N_CATEGORICALS' in self.config or 'N_CLASSES' in self.config or 'DETERMINISTIC' in self.config:
            cfg.STOCHASTIC = cfg.N_CATEGORICALS * cfg.N_CLASSES
            cfg.FEAT = cfg.STOCHASTIC + cfg.DETERMINISTIC
        return cfg

    def _build(self, obs_dim: int):
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)
        LearnerConfig, ControllerConfig, _ = self._configs
        self.obs_dim = self.kwargs['obs_dim'] = obs_dim
        self.learner_config = self._apply(LearnerConfig())
        self.controller_config = self._apply(ControllerConfig(), strict=False)   # the model's settings
        self.learner = self.learner_config.create_learner()

    def controller(self):
        "A controller (MAMBA's inference: one env's recurrent states) with the current parameters."
        controller = self.controller_config.create_controller()
        controller.receive_params(self.learner.params())
        return controller

    def _observe(self, info: dict, goal: np.ndarray | None) -> np.ndarray:
        return agent_features(info, 0, self.obs_keys, self.grid_size, goal if self.goal_conditioned else None)

    def train(self, world, episodes: int, seed: int = 0, log_every: int = 10,
              log: Callable[[dict], None] | None = None) -> dict[str, list]:
        """
        Play and learn from `episodes` episodes of `world` (one env), episode ``k`` reset with seed
        ``seed + k``. Every `log_every` episodes, the success rate, mean episode length and return
        (the env's), and MAMBA's latest losses go to ``history`` (and to `log`).
        """
        from stable_marl.wm.mamba.mamba_marl.log import wandb
        if world.num_envs != 1:
            raise ValueError("Mamba.train plays one env at a time: make the World with num_envs=1")
        history: dict[str, list] = {}
        window = []
        controller = None
        for k in range(episodes):
            if self.learner is None:
                world.reset(seed=seed)
                self._build(self._observe(world.infos, world.infos['goal_position'][0, 0]).shape[1])
            if controller is None:
                controller = self.controller()
            rollout, stats, episode = self._play(world, controller, seed + k)
            self.learner.step(rollout)
            if self.goal_conditioned:
                for copy in self._relabel(episode):
                    self.learner.replay_buffer.append(*copy, None)
            controller.receive_params(self.learner.params())
            window.append(stats)
            if (k + 1) % log_every == 0 or k + 1 == episodes:
                row = {'episode': k + 1, 'env_steps': self.learner.total_samples,
                       'success_rate': 100.0 * float(np.mean([s['success'] for s in window])),
                       'mean_length': float(np.mean([s['length'] for s in window])),
                       'mean_return': float(np.mean([s['return'] for s in window])),
                       **{f'mamba/{key}': v for key, v in wandb.latest.items()}}
                for key, v in row.items():
                    history.setdefault(key, []).append(v)
                window = []
                if log: log(row)
        return history

### One episode

As MAMBA's worker: each step stores the agents' observations (absorbing zeros, ``fake`` = 1, for agents
that already finished), actions (one-hot), rewards and ``done`` flags; when every agent finished (not at
the step limit), two absorbing steps close the episode. The returned episode keeps what hindsight
relabelling needs.

In [ ]:
#| export
def _shaping_distances(env) -> np.ndarray:
    "Shortest-path distance of each agent to the goal (0 for agents that finished)."
    from stable_marl.envs.multigrid.expert import goal_distances
    distance = goal_distances(env)
    out = []
    for agent in env.agents:
        x, y = (int(v) for v in agent.state.pos)
        inside = 0 <= x < distance.shape[0] and 0 <= y < distance.shape[1]
        d = distance[x, y] if inside and not agent.state.terminated else 0.0
        out.append(d if np.isfinite(d) else 1e3)   # MAMBA's _NEARINF for unreachable cells
    return np.array(out, np.float32)


def _absorbing_rows(controller, num_agents: int, num_actions: int, obs_dim: int):
    "MAMBA's two closing steps: absorbing observations, random actions, ``fake`` = ``done`` = 1."
    for _ in range(2):
        action = torch.zeros(1, num_agents, num_actions)
        action.scatter_(2, torch.randint(0, num_actions, (1, num_agents, 1)), 1.0)
        controller.update_buffer({'observation': torch.zeros(1, num_agents, obs_dim), 'action': action,
                                  'reward': torch.zeros(1, num_agents, 1), 'fake': torch.ones(1, num_agents, 1),
                                  'done': torch.ones(1, num_agents, 1)})


from fastcore.utils import patch

@patch
def _play(self: Mamba, world, controller, seed: int):
    A, K = self.num_agents, self.num_actions
    world.reset(seed=seed)
    env = world.envs.envs[0].unwrapped
    goal = world.infos['goal_position'][0, 0].copy()
    controller.init_rnns()
    finished = np.zeros(A, bool)
    distance = _shaping_distances(env) if not self.goal_conditioned else None
    episode = {'base': [], 'position': [], 'action': [], 'achieved': [], 'finished': []}
    total, length, success = 0.0, 0, False
    for _ in itertools.count():
        obs = self._observe(world.infos, goal)
        obs[finished] = 0.0                                             # absorbing observations
        if self.goal_conditioned:
            episode['base'].append(agent_features(world.infos, 0, self.obs_keys, self.grid_size))
            episode['position'].append(world.infos['position'][0, 0].astype(np.float32))
            episode['finished'].append(finished.copy())
        action = controller.step(torch.as_tensor(obs)[None], None, None)   # (A, K) one-hot
        _, rewards, terms, truncs, world.infos = world.envs.step(action.argmax(-1).numpy()[None])
        env_reward, term, trunc = rewards[0], terms[0].astype(bool), bool(truncs[0].any())
        reached = term & ~finished
        reward = self.finish_value * reached.astype(np.float32)
        if not self.goal_conditioned:                                    # MAMBA's Flatland rewards
            new_distance = _shaping_distances(env)
            reward = reward + env_reward + self.near_coeff * (distance - new_distance)
            distance = new_distance
        controller.update_buffer({'observation': torch.as_tensor(obs)[None], 'action': action[None],
                                  'reward': torch.as_tensor(reward, dtype=torch.float32)[None, :, None],
                                  'done': torch.as_tensor(term, dtype=torch.float32)[None, :, None],
                                  'fake': torch.as_tensor(finished, dtype=torch.float32)[None, :, None]})
        if self.goal_conditioned:
            episode['action'].append(action.numpy())
            episode['achieved'].append(world.infos['position'][0, 0].astype(np.float32))
        total += float(np.sum(env_reward))
        length += 1
        finished = term.copy()
        if finished.all():
            success = True
            _absorbing_rows(controller, A, K, self.obs_dim)
            break
        if trunc:
            break
    return controller.dispatch_buffer(), {'success': success, 'length': length, 'return': total}, episode

### Hindsight relabelling (GC-MAMBA)

Each copy gives every agent a goal it reached later in the episode (a cell, as the env's goals), then
replays the episode with that goal: the agent is rewarded and done when it first stands on it, and
absorbing afterwards. The copy ends with MAMBA's two absorbing steps.

In [ ]:
#| export
@patch
def _relabel(self: Mamba, episode: dict) -> list[tuple]:
    "`relabel_k` hindsight copies of `episode`, as replay-buffer rows (obs, action, reward, done, fake, last)."
    if not episode['action']:
        return []
    base, pos = np.stack(episode['base']), np.stack(episode['position'])            # (T, A, D), (T, A, 2)
    actions, achieved = np.stack(episode['action']), np.stack(episode['achieved'])  # (T, A, K), (T, A, 2)
    was_finished = np.stack(episode['finished'])                                    # (T, A): before step t
    T, A = actions.shape[:2]
    copies = []
    for _ in range(self.relabel_k):
        goal, stop = np.zeros((A, 2), np.float32), np.zeros(A, int)
        for a in range(A):
            alive = np.flatnonzero(~was_finished[:, a])                              # steps the agent acted
            k = np.random.choice(alive)
            goal[a] = achieved[k, a]
            stop[a] = np.flatnonzero((achieved[:, a] == goal[a]).all(-1))[0]          # first time on the goal
        n = stop.max() + 1
        obs = np.zeros((n + 2, A, self.obs_dim), np.float32)
        act = np.zeros((n + 2, A, self.num_actions), np.float32)
        rew, done, fake = (np.zeros((n + 2, A, 1), np.float32) for _ in range(3))
        for t in range(n):
            for a in range(A):
                act[t, a] = actions[t, a]
                if t <= stop[a]:
                    obs[t, a] = np.concatenate([base[t, a], (goal[a] - pos[t, a]) / self.grid_size])
                    rew[t, a, 0] = self.finish_value * (t == stop[a])
                    done[t, a, 0] = float(t == stop[a])
                else:
                    fake[t, a, 0] = done[t, a, 0] = 1.0
        act[n:, np.arange(A), np.random.randint(0, self.num_actions, (2, A))] = 1.0
        fake[n:], done[n:] = 1.0, 1.0
        last = np.zeros((n + 2, A, 1), np.float32)
        last[-1] = 1.0
        copies.append((obs, act, rew, done, fake, last))
    return copies

### Acting and checkpoints

`MambaPolicy` runs one controller per env of a `World` (recurrent states reset at each new episode),
with MAMBA's sampled actions (or the most likely ones, `deterministic`). MAMBA's latent states are
sampled too (one-hot categoricals, at inference as well), so runs are reproducible only with torch's seed.

In [ ]:
#| export
class MambaPolicy(BasePolicy):
    "Actions of a trained :class:`Mamba` for every env of a World."
    def __init__(self, mamba: Mamba, deterministic: bool = False, **kwargs: Any):
        super().__init__(**kwargs)
        self.type = 'mamba'
        self.mamba, self.deterministic = mamba, deterministic
        self._controllers: list = []

    def set_env(self, env: Any):
        super().set_env(env)
        self._controllers = [self.mamba.controller() for _ in range(env.num_envs)]

    @torch.no_grad()
    def _step(self, controller, obs: torch.Tensor) -> torch.Tensor:
        if not self.deterministic:
            return controller.step(obs, None, None)
        state = controller.model(obs, controller.prev_actions, controller.prev_rnn_state, None)
        _, logits = controller.actor(state.get_features())
        action = torch.nn.functional.one_hot(logits.argmax(-1), logits.shape[-1]).float()
        controller.advance_rnns(state)
        controller.prev_actions = action.clone()
        return action.squeeze(0)

    def get_action(self, obs: dict, **kwargs: Any) -> np.ndarray:
        m = self.mamba
        flush = obs.get('_needs_flush')
        actions = []
        for i, controller in enumerate(self._controllers):
            if flush is not None and flush[i]:
                controller.init_rnns()
            goal = obs['goal_position'][i, 0] if m.goal_conditioned else None
            x = agent_features(obs, i, m.obs_keys, m.grid_size, goal)
            x[np.asarray(obs['terminated'][i, 0], bool)] = 0.0          # finished agents: absorbing
            actions.append(self._step(controller, torch.as_tensor(x)[None]).argmax(-1).numpy())
        return np.stack(actions)


@patch
def policy(self: Mamba, deterministic: bool = False) -> MambaPolicy:
    "A :class:`MambaPolicy` with the current parameters."
    return MambaPolicy(self, deterministic)


@patch
def save(self: Mamba, path: str | Path, **extra: Any) -> Path:
    "Save the settings and the learner's parameters (model, actor, critic) to `path`; `extra` is stored too."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'kwargs': self.kwargs, 'params': self.learner.params(), **extra}, path)
    return path


@patch(cls_method=True)
def load(cls: Mamba, path: str | Path) -> Mamba:
    state = torch.load(path, map_location='cpu', weights_only=False)
    mamba = cls(**state['kwargs'])
    for name in ('model', 'actor', 'critic'):
        getattr(mamba.learner, name).load_state_dict(state['params'][name])
    mamba.checkpoint = {k: v for k, v in state.items() if k not in ('kwargs', 'params')}
    return mamba

### Tests

In [ ]:
import tempfile
from fastcore.test import test_fail
from stable_marl import World
from stable_marl.envs.multigrid import GoToGoalPolicy

TINY = dict(HIDDEN=32, MODEL_HIDDEN=32, EMBED=32, N_CATEGORICALS=4, N_CLASSES=4, DETERMINISTIC=32,
            VALUE_HIDDEN=32, PCONT_HIDDEN=32, ACTION_HIDDEN=32, REWARD_HIDDEN=32,
            MIN_BUFFER_SIZE=40, SEQ_LENGTH=10, BATCH_SIZE=4, MODEL_BATCH_SIZE=4, PPO_EPOCHS=1, CAPACITY=5000)
kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=1, goal_conditioned=True, max_episode_steps=12, **kw)

# observations: symbolic local view (7 x 7 x 3) + heading (4), plus the goal offset for GC-MAMBA
world.reset(seed=0)
x = agent_features(world.infos, 0, ['image', 'direction'], 7)
assert x.shape == (2, 151) and x[:, 147:].sum(1).tolist() == [1.0, 1.0]
g = agent_features(world.infos, 0, ['image', 'direction'], 7, goal=world.infos['goal_position'][0, 0])
assert g.shape == (2, 153) and np.allclose(g[:, -2:] * 7, world.infos['goal_position'][0, 0, :] - world.infos['position'][0, 0])

for goal_conditioned in (False, True):
    m = Mamba(2, 4, goal_conditioned=goal_conditioned, grid_size=7, relabel_k=2, seed=0, **TINY)
    hist = m.train(world, episodes=6, log_every=3)
    assert hist['episode'] == [3, 6] and m.learner.total_samples > 40, hist
    assert any(k.startswith('mamba/Model') for k in hist) or any(k.startswith('mamba/') for k in hist)
    if goal_conditioned:   # hindsight copies end with the agents on their relabelled goals
        assert len(m.learner.replay_buffer) > m.learner.total_samples
    w = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, max_episode_steps=12, **kw)
    for det in (False, True):
        w.set_policy(m.policy(deterministic=det))
        assert len(w.evaluate(episodes=3, seed=10)['episode_successes']) == 3
    # save / load: the same deterministic actions
    with tempfile.TemporaryDirectory() as tmp:
        loaded = Mamba.load(m.save(f'{tmp}/m.pt', env_name='x'))
        assert loaded.checkpoint == {'env_name': 'x'}
        w.set_policy(m.policy(deterministic=True)); w.reset(seed=4)
        torch.manual_seed(0); a = w.policy.get_action(dict(w.infos))
        w.set_policy(loaded.policy(deterministic=True)); w.reset(seed=4)
        torch.manual_seed(0)                                    # the latent states are sampled
        assert np.array_equal(w.policy.get_action(dict(w.infos)), a)

# the hindsight copies: each agent is rewarded and done once, on the step it reaches its relabelled goal
m = Mamba(2, 4, goal_conditioned=True, grid_size=7, relabel_k=3, **TINY)
world.reset(seed=0); m._build(153)
_, stats, episode = m._play(world, m.controller(), seed=1)
for obs, act, rew, done, fake, last in m._relabel(episode):
    assert obs.shape[1:] == (2, 153) and last[-1].all() and last[:-1].sum() == 0
    for a in range(2):
        hit = np.flatnonzero(rew[:, a, 0] > 0)
        assert len(hit) == 1 and done[hit[0], a, 0] == 1 and done[:hit[0], a, 0].sum() == 0
        assert (fake[hit[0] + 1:, a] == 1).all() and (obs[hit[0] + 1:, a] == 0).all()
test_fail(lambda: Mamba(2, 4, NOPE=1, obs_dim=5), contains='NOPE')
test_fail(lambda: m.train(World('MultiGrid-FindGoal-15x15-v0', num_envs=2, **kw), 1), contains='num_envs=1')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()